# Home Credit Default Risk — Milestone 1b: LightGBM vs XGBoost Comparison

**Purpose:** run the same baseline pipeline from `milestone1_baseline.ipynb`, but train
**both LightGBM and XGBoost** on the identical train/validation split, so their scores
are directly comparable (same data, same split, same random seed).

**This notebook covers:**
1. Load & inspect data (same as Milestone 1)
2. Handle missing values (same as Milestone 1)
3. Encode categorical features (same as Milestone 1)
4. Fix the `DAYS_EMPLOYED` data quality issue (same as Milestone 1)
5. Train/validation split (same seed as Milestone 1, so results are comparable)
6. Train Logistic Regression, LightGBM, and XGBoost
7. Final comparison table: AUC + training time for all three

**Repo structure note:** this notebook is designed to live in the `notebooks/` folder of the
project repo, reading data from `../data/raw/`. Run it locally with:
`uv run jupyter notebook` (or however you normally launch notebooks in this project) —
it needs `xgboost` installed (`uv add xgboost` or `pip install xgboost`).


## 1. Load & Inspect Data

In [1]:
import pandas as pd
import numpy as np
import time

DATA_PATH = "../data/raw/application_train.csv"

df = pd.read_csv(DATA_PATH)
sk_id = df['SK_ID_CURR']

print("Shape:", df.shape)
print("\nTarget distribution (%):")
print((df['TARGET'].value_counts(normalize=True) * 100).round(2))

Shape: (307511, 122)

Target distribution (%):
TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64


## 2. Handle Missing Values

In [2]:
y = df['TARGET']
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])

missing_frac = X.isnull().mean().sort_values(ascending=False)

# Same override as Milestone 1: keep EXT_SOURCE_1 despite >50% missing --
# it's one of the strongest predictors in the dataset.
keep_despite_missing = ['EXT_SOURCE_1']

cols_to_drop = missing_frac[missing_frac > 0.5].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c not in keep_despite_missing]

X = X.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns with >50% missing (kept {keep_despite_missing})")
print("Shape after dropping:", X.shape)

Dropped 40 columns with >50% missing (kept ['EXT_SOURCE_1'])
Shape after dropping: (307511, 80)


## 3. Encode Categorical Features

In [3]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

print(f"Categorical columns: {len(cat_cols)}, Numeric columns: {len(num_cols)}")

for col in cat_cols:
    X[col] = X[col].fillna('missing')

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print("Shape after encoding:", X.shape)

Categorical columns: 13, Numeric columns: 67
Shape after encoding: (307511, 80)


## 4. Fix Data Quality Issue: `DAYS_EMPLOYED`

Same fix as Milestone 1: `365243` is a disguised "not currently employed" placeholder,
not a real duration. Replace with `np.nan` and add a flag column.

In [4]:
X['DAYS_EMPLOYED_ANOM'] = (X['DAYS_EMPLOYED'] == 365243).astype(int)
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

print("DAYS_EMPLOYED dtype:", X['DAYS_EMPLOYED'].dtype)
print(f"Anomaly flag positive rate: {X['DAYS_EMPLOYED_ANOM'].mean()*100:.2f}%")

DAYS_EMPLOYED dtype: float64
Anomaly flag positive rate: 18.01%


## 5. Train / Validation Split

Same `random_state=42` and stratified split as Milestone 1 — this is what makes the
comparison fair: every model below trains and validates on exactly the same rows.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Train shape: (246008, 81)
Val shape: (61503, 81)


## 6a. Logistic Regression (reference baseline)

Kept from Milestone 1 as the simple, interpretable reference point.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

train_medians = X_train.median()
X_train_filled = X_train.fillna(train_medians)
X_val_filled = X_val.fillna(train_medians)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_val_scaled = scaler.transform(X_val_filled)

t0 = time.time()
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)
logreg_train_time = time.time() - t0

val_pred_lr = logreg.predict_proba(X_val_scaled)[:, 1]
auc_lr = roc_auc_score(y_val, val_pred_lr)
print(f"Logistic Regression Validation AUC: {auc_lr:.4f}  (train time: {logreg_train_time:.1f}s)")

Logistic Regression Validation AUC: 0.7472  (train time: 3.8s)


## 6b. LightGBM (existing Milestone 1 model, for comparison)

Same params as Milestone 1, so this reproduces the original ~0.7619 AUC exactly.

In [7]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'is_unbalance': True,
    'verbosity': -1,
    'seed': 42
}

t0 = time.time()
model_lgb = lgb.train(
    lgb_params, lgb_train,
    valid_sets=[lgb_val],
    num_boost_round=200,
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
)
lgb_train_time = time.time() - t0

val_pred_lgb = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
auc_lgb = roc_auc_score(y_val, val_pred_lgb)
print(f"LightGBM Validation AUC: {auc_lgb:.4f}  (best iter: {model_lgb.best_iteration}, train time: {lgb_train_time:.1f}s)")

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[171]	valid_0's auc: 0.761933
LightGBM Validation AUC: 0.7619  (best iter: 171, train time: 7.5s)


## 6c. XGBoost (new model)

Same role as LightGBM above — a gradient-boosted tree model — using equivalent settings
so the comparison is about the *algorithm*, not mismatched hyperparameters:
- `scale_pos_weight` mirrors LightGBM's `is_unbalance=True` (compensates for the ~92/8 imbalance)
- Same number of boosting rounds and early stopping patience
- XGBoost's histogram method (`tree_method='hist'`) is the closest match to LightGBM's approach

In [8]:
import xgboost as xgb

# scale_pos_weight = (# negative) / (# positive), the XGBoost equivalent of is_unbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'scale_pos_weight': scale_pos_weight,
    'seed': 42
}

t0 = time.time()
model_xgb = xgb.train(
    xgb_params, dtrain,
    evals=[(dval, 'val')],
    num_boost_round=200,
    early_stopping_rounds=20,
    verbose_eval=False
)
xgb_train_time = time.time() - t0

val_pred_xgb = model_xgb.predict(dval, iteration_range=(0, model_xgb.best_iteration + 1))
auc_xgb = roc_auc_score(y_val, val_pred_xgb)
print(f"XGBoost Validation AUC: {auc_xgb:.4f}  (best iter: {model_xgb.best_iteration}, train time: {xgb_train_time:.1f}s)")

XGBoost Validation AUC: 0.7539  (best iter: 38, train time: 5.1s)


## 7. Feature Importance — LightGBM vs XGBoost

In [9]:
lgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'lgb_importance': model_lgb.feature_importance(importance_type='gain')
}).sort_values('lgb_importance', ascending=False)

xgb_score_dict = model_xgb.get_score(importance_type='gain')
xgb_importance = pd.DataFrame({
    'feature': list(xgb_score_dict.keys()),
    'xgb_importance': list(xgb_score_dict.values())
}).sort_values('xgb_importance', ascending=False)

print("Top 8 features -- LightGBM (gain):")
print(lgb_importance.head(8).to_string(index=False))
print("\nTop 8 features -- XGBoost (gain):")
print(xgb_importance.head(8).to_string(index=False))

Top 8 features -- LightGBM (gain):
        feature  lgb_importance
   EXT_SOURCE_3   201679.566411
   EXT_SOURCE_2   152710.905025
   EXT_SOURCE_1    66457.644361
  DAYS_EMPLOYED    42426.502665
     AMT_CREDIT    32496.576123
     DAYS_BIRTH    25913.790261
    AMT_ANNUITY    25289.808619
AMT_GOODS_PRICE    24973.222741

Top 8 features -- XGBoost (gain):
            feature  xgb_importance
       EXT_SOURCE_2      300.966888
       EXT_SOURCE_3      291.676514
NAME_EDUCATION_TYPE      233.697067
       FLAG_OWN_CAR      146.360489
        CODE_GENDER      140.005630
 NAME_CONTRACT_TYPE      127.693474
       EXT_SOURCE_1      119.972260
    FLAG_DOCUMENT_3       99.257042


## 8. Final Comparison

Same data, same split, same imbalance-handling approach for both boosting models --
so any AUC difference here reflects the algorithms themselves, not mismatched setup.

In [10]:
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM', 'XGBoost'],
    'Validation AUC': [auc_lr, auc_lgb, auc_xgb],
    'Train Time (s)': [round(logreg_train_time, 1), round(lgb_train_time, 1), round(xgb_train_time, 1)]
})
print(summary.to_string(index=False))

              Model  Validation AUC  Train Time (s)
Logistic Regression        0.747189             3.8
           LightGBM        0.761933             7.5
            XGBoost        0.753876             5.1


**Milestone 1b status: LightGBM vs XGBoost comparison complete.**

Fill in after running:
- Validation AUC — LightGBM: `____`  vs  XGBoost: `____`
- Training time — LightGBM: `____`s  vs  XGBoost: `____`s
- Which model won, and by how much?
- Did the top features roughly agree between the two? (They usually do, since both are
  tree-based boosting models learning from the same signal.)

**Note:** since `data/raw/application_train.csv` isn't available in this environment, this
notebook was written but not executed here — run it locally to get your real numbers.
